# Avance 4 · Regresión de la potencia en la fase 2

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04_Models](04_Models.ipynb) para la fase 2. Allí la comparativa de seis modelos es de **clasificación a 10 µM** y queda como el piso; aquí se mide lo que la fase 2 predice ahora: la **potencia** (`pActividad`) de un compuesto sobre una diana, y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)).

Son seis modelos individuales (sin ensambles), con el mismo protocolo que el 04. Se entrenan con las mediciones exactas: el modelo con censura es el baseline del [03b](03b_Baseline_regresion_actividad.ipynb), y su evidencia es lo que decide qué viene en el Avance 5.


## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import concordancia_harrell, rmse_exactas

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### Los datos

La unidad es el par compuesto-diana. Se usan los mismos pares de entrenamiento y validación que el 04 (la prueba reservada no se toca) y la diana entra como columnas de 0 y 1. Las filas son las del baseline del 03b: **ninguna se descarta por no decidir a 10 µM**, y salen solo las que no informan nada (sin ninguna cota, o con cotas que se contradicen). Los modelos se entrenan con las exactas; la validación incluye las censuradas, porque la concordancia las usa.

In [2]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")

56,411 pares de entrenamiento y validación · 170 sin información y salen
56,241 pares · 46,100 con potencia exacta (82.0%) · 10,141 solo con cotas


## 2. Seis modelos de regresión

| Clasificación (notebook 04) | Regresión (aquí) |
|:--|:--|
| logística | `Ridge` (regresión lineal) |
| árbol | `DecisionTreeRegressor` |
| kNN | `KNeighborsRegressor` |
| Naive Bayes | `BayesianRidge` (regresión lineal bayesiana) |
| SVM-RBF | `SVR` |
| MLP | `MLPRegressor` |

Naive Bayes no tiene versión de regresión; `BayesianRidge` es la familia bayesiana que sí la tiene. `media` (predecir siempre el promedio de las exactas) va aparte como **referencia**: es el piso y no cuenta entre los seis, porque no es un algoritmo.

**Métrica principal, fijada antes de ver los resultados: la concordancia de Harrell.** Es la única que usa también las mediciones censuradas, y la tabla se ordena por ella. Las otras son el Spearman y el RMSE sobre las exactas (declarado así: no dicen nada de las censuradas), y el MCC y el AUROC de la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlos con el 04.

**El mínimo que hay que superar** viene del [03b](03b_Baseline_regresion_actividad.ipynb): el mejor de cada métrica entre sus baselines, concordancia 0.700, Spearman 0.531 y RMSE 1.156. Un modelo que no los supere no aporta sobre una recta.

La validación es la del 04: cinco pliegues agrupados por `grupo_cv`, semillas 0, 1 y 2, que también fijan el azar de cada modelo.

In [3]:
MODELOS = {
    "ridge":          make_pipeline(MinMaxScaler(), Ridge()),
    "arbol":          DecisionTreeRegressor(min_samples_leaf=5),
    "knn":            make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "bayesian_ridge": make_pipeline(StandardScaler(), BayesianRidge()),
    "svr":            make_pipeline(MinMaxScaler(), SVR()),
    "mlp":            make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
REFERENCIA = {"media": DummyRegressor()}        # el piso: no aprende nada
SEMILLAS = (0, 1, 2)
CORTE = 5.0                                      # pActividad 5 = 10 µM, el mismo corte del 04
MINIMOS = {"concordancia": 0.700, "spearman": 0.531, "rmse": 1.156}     # del 03b


def medir(inf, sup, clase, pred):
    """Concordancia con todas las mediciones; Spearman y RMSE con las exactas; MCC y AUROC de la clase derivada."""
    exa = inf == sup
    pe = pred[exa]
    rho = np.nan if np.ptp(pe) == 0 else spearmanr(inf[exa], pe).statistic   # nan si predice una constante
    rmse, _ = rmse_exactas(inf, sup, pred)
    fila = {"concordancia": concordancia_harrell(inf, sup, pred), "spearman": rho, "rmse": rmse,
            "mcc": np.nan, "auroc": np.nan}
    decide = ~np.isnan(clase)
    if len(np.unique(clase[decide])) == 2:
        fila["mcc"] = matthews_corrcoef(clase[decide], pred[decide] >= CORTE)
        fila["auroc"] = roc_auc_score(clase[decide], pred[decide])
    return fila


filas = []
estratificar = np.where(np.isnan(y_cls), -1, y_cls).astype(int)
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, estratificar, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]                       # se entrena solo con las exactas
        for nombre, modelo in {**MODELOS, **REFERENCIA}.items():
            m = clone(modelo)
            # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
            m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], inferior[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            base = {"modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
            filas.append({**base, "estrato": "todos",
                          **medir(inferior[val], superior[val], y_cls[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                ys = ys[~np.isnan(ys)]
                if min(ys.sum(), (1 - ys).sum()) >= 20:         # estrato con 20 casos de cada clase
                    filas.append({**base, "estrato": e,
                                  **medir(inferior[val][sel], superior[val][sel], y_cls[val][sel], pred[sel])})
res = pd.DataFrame(filas)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × {len(SEMILLAS)} semillas × 5 pliegues, por estrato")

420 filas: 7 modelos × 3 semillas × 5 pliegues, por estrato


## 3. Resultados

In [4]:
def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por la métrica principal. `desv_concordancia` es dispersión entre
    evaluaciones, no error estándar: las semillas comparten filas."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["concordancia", "spearman", "rmse", "mcc", "auroc", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby("modelo")["concordancia"].std())
    t["supera_minimos"] = ((t["concordancia"] > MINIMOS["concordancia"]).astype(int)
                           + (t["spearman"] > MINIMOS["spearman"]).astype(int)
                           + (t["rmse"] < MINIMOS["rmse"]).astype(int))
    return t.sort_values("concordancia", ascending=False).round(3)


tabla_todos = tabla(res)
tabla_todos

,concordancia,desv_concordancia,spearman,rmse,mcc,auroc,seg,supera_minimos
modelo,,,,,,,,
knn,0.739,0.004,0.660,1.010,0.341,0.805,0.012,3
svr,0.738,0.004,0.640,1.034,0.353,0.819,50.741,3
mlp,0.735,0.006,0.640,1.092,0.370,0.800,16.647,3
bayesian_ridge,0.692,0.006,0.530,1.157,0.225,0.776,0.119,0
ridge,0.691,0.006,0.528,1.148,0.222,0.774,0.022,1
arbol,0.670,0.007,0.484,1.308,0.286,0.717,2.047,0
media,0.500,0.000,NaN,1.337,0.000,0.500,0.004,0


### Por procedencia

Un promedio puede esconder cómo le va a los naturales. En el 03b la concordancia de las rectas sube de 0.53 a 0.64 solo en los pares que solo tiene NPASS, y el control del Avance 3 mostró que el modelo separa mucho peor esos pares que los de ChEMBL.

In [5]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "desv_concordancia", "spearman", "rmse", "mcc", "auroc"]]

concordancia  desv_concordancia  spearman   rmse  \
procedencia modelo                                                             
solo_chembl knn                    0.733              0.004     0.644  1.013   
            svr                    0.730              0.005     0.620  1.040   
            mlp                    0.729              0.006     0.627  1.091   
            bayesian_ridge         0.681              0.007     0.502  1.148   
            ridge                  0.680              0.007     0.501  1.151   
            arbol                  0.665              0.007     0.468  1.312   
            media                  0.500              0.000       NaN  1.325   
mixto       mlp                    0.570              0.024     0.203  1.145   
            knn                    0.565              0.025     0.294  0.981   
            svr                    0.564              0.022     0.211  0.988   
            arbol                  0.532              0.016     0.104  1.266   
            ridge                  0.515              0.030     0.052  1.091   
            bayesian_ridge         0.510              0.031     0.044  1.092   
            media                  0.500              0.000       NaN  1.498   
solo_npass  mlp                    0.551              0.031     0.249  1.065   
            svr                    0.535              0.028     0.313  0.813   
            bayesian_ridge         0.532              0.029     0.240  1.292   
            ridge                  0.522              0.028     0.236  1.056   
            media                  0.500              0.000       NaN  1.640   
            arbol                  0.498              0.037     0.194  1.175   
            knn                    0.464              0.019     0.342  0.859   

                              mcc  auroc  
procedencia modelo                        
solo_chembl knn             0.320  0.798  
            svr             0.318  0.808  
            mlp             0.345  0.793  
            bayesian_ridge  0.172  0.762  
            ridge           0.169  0.761  
            arbol           0.269  0.711  
            media           0.000  0.500  
mixto       mlp             0.142  0.598  
            knn             0.138  0.615  
            svr             0.128  0.601  
            arbol           0.072  0.537  
            ridge          -0.006  0.507  
            bayesian_ridge -0.003  0.501  
            media           0.000  0.500  
solo_npass  mlp             0.058  0.571  
            svr             0.015  0.542  
            bayesian_ridge  0.013  0.528  
            ridge           0.014  0.517  
            media           0.000  0.500  
            arbol           0.003  0.512  
            knn            -0.013  0.490

## 3.1 Lectura

Escrita después de ver la tabla. La corrida es de Artemio Padilla, en macOS (arm64) con Python 3.12.12 y las versiones del lock; la primera celda lo imprime. No se tocó el código del cuaderno.

**Lo que establece.**

1. **Tres familias superan los tres mínimos del 03b y se empatan entre sí.** kNN (concordancia 0.739, Spearman 0.660, RMSE 1.010), SVR (0.738, 0.640, 1.034) y MLP (0.735, 0.640, 1.092). Entre ellas la diferencia de concordancia, de 0.001 a 0.004, es menor que la dispersión entre evaluaciones (0.004 a 0.006): por la métrica principal no hay un ganador. Las dos rectas se quedan donde estaba la recta con exactas del 03b (`ridge` 0.691, `bayesian_ridge` 0.692, contra 0.693) y el árbol en 0.670.
2. **La flexibilidad paga en ChEMBL y no en NPASS.** En los pares de solo ChEMBL, kNN, SVR y MLP sacan de 0.729 a 0.733 de concordancia contra 0.680 y 0.681 de las rectas. En los de solo NPASS, **las seis familias están cerca del azar**: de 0.464 a 0.551 de concordancia (el kNN queda por debajo del piso de 0.500), MCC de −0.013 a 0.058 y AUROC de 0.490 a 0.571. En los mixtos, de 0.510 a 0.570.
3. **Entrenar solo con exactas es el límite, no la familia.** El baseline con censura del 03b, una recta, saca 0.639 en el mismo estrato; la mejor de las seis aquí, el MLP, 0.551. La diferencia, 0.088, son unas 2.8 veces la dispersión de ese estrato (0.031). La comparación es indicativa y no de la misma corrida: el 03b usó cinco semillas y Linux, esta tabla tres y macOS.
4. **Ordenar bien las exactas no basta.** En solo NPASS el kNN y el SVR tienen el menor RMSE (0.859 y 0.813) y los mayores Spearman (0.342 y 0.313) de la tabla, pero la concordancia más baja. Una lectura probable, que este cuaderno no demuestra: predicen bien las mediciones exactas y colocan mal las censuradas, que son la mayor parte de los negativos de NPASS (1,208 pares dependen solo de `AC50`, con cota).
5. **Ninguna se acerca al techo de ruido** (RMSE de 1.010 a 1.308 contra 0.7), ni hay señal de fuga.
6. **Tiempos por ajuste, con un solo hilo y nada más corriendo:** kNN 0.012 s, `ridge` 0.022 s, `bayesian_ridge` 0.12 s, árbol 2.0 s, MLP 16.6 s y SVR 50.7 s. Son de un procesador de Apple y no son comparables con los del 04, que se midieron en Docker.

**Lo que no establece.**

- **No elige un modelo.** Los tres primeros no se distinguen, y la elección de la fase 2 depende de si se acepta entrenar con censuradas, que es el Avance 5.
- **No dice que la censura sea la causa:** el 03b y esta corrida difieren en semillas, sistema y modelo. Lo que sí dice es que ninguna de las seis, entrenada con exactas, recupera a los naturales.
- **El MCC y el AUROC no son comparables con los del 04**: son otro modelo y se calculan sobre los pares que deciden a 10 µM.
- **Tres semillas que comparten filas:** la dispersión no es un error estándar, y el estrato de solo NPASS tiene 2,114 pares, de los cuales solo 982 tienen una medición exacta.

**Pendiente, para el Avance 5.** Las sensibilidades declaradas de antemano en la [página de variables objetivo](../docs/variable-objetivo-fase2.md) —sin las relaciones inferidas, solo dirección consistente, con y sin DrugMatrix— y el corte de 1 µM junto al de 10 µM. **Lo de DrugMatrix, en particular, no se ha medido todavía.**


In [6]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("comparativa-regresion-seis-modelos", decision="2.9",
             params={"modelos": list(MODELOS), "referencia": list(REFERENCIA), "pliegues": 5,
                     "entrena_con": "solo exactas", "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "corte_mcc": CORTE,
                     "minimos_03b": MINIMOS, "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

2026-10-25-artemio-2.9-comparativa-regresion-seis-modelos


# Declaración de uso de IA y contribución

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Revisar este cuaderno sobre la versión de Antonio: sexto modelo, métricas de regresión, el criterio de las filas, el mínimo del 03b, la corrida registrada y la lectura | Todas | **Lo que el equipo cambió:** `media` sale de los seis, porque no es un algoritmo, y entra `BayesianRidge`; las métricas pasan de MAE a concordancia, Spearman y RMSE sobre las exactas; las filas ya no se filtran por decidir a 10 µM; el kernel pasa a `python3` y se corrigen las erratas. La ejecución la hizo Artemio Padilla en macOS (Python 3.12.12, versiones del lock) sin tocar el código; la lectura de la sección 3.1 la escribió Ingrid Pamela Ruiz Puga con Claude Code a partir de esa corrida, y la revisan ella y José Antonio Vázquez Martínez (por confirmar en la revisión del PR) |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | |
| Artemio Santiago Padilla Robles | |
| José Antonio Vázquez Martínez | |

Las salidas de la herramienta se usaron como insumo de trabajo y el equipo asume la responsabilidad por la exactitud, la integridad y el contenido final del cuaderno.
